In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import sys; sys.path.append('/content/drive/MyDrive/CS3268_Project/src')
!pip install -q fairlearn
import os, numpy as np, pandas as pd, xgboost as xgb
from sklearn.base import BaseEstimator, ClassifierMixin
from fairlearn.reductions import ExponentiatedGradient, FalsePositiveRateParity
from common import load_split, features, train_model, threshold_at_fpr, BEST_PARAMS, LABEL, ROOT, PREDS, CATEGORICALS
np.random.seed(42)
OUT = f"{ROOT}/models/expgrad_robust_db0.05"; os.makedirs(OUT, exist_ok=True)

In [ ]:
# Data: Base train + Variant II train (categoricals aligned to the BASE category sets)
base_tr = load_split("train")
cats = {c: base_tr[c].cat.categories for c in CATEGORICALS}
def align(df):
    df = df.copy()
    for c in CATEGORICALS: df[c] = df[c].astype(pd.CategoricalDtype(cats[c]))
    return df
v2 = {s: align(load_split(s, "variant2")) for s in ("train", "val", "test")}
trc = pd.concat([base_tr, v2["train"]], ignore_index=True)
Xc, yc, gc = features(trc), trc[LABEL], trc["age_group"]
print(len(trc))   # expect 1,351,332 (675,666 x 2 if Variant II has the same split sizes)

In [ ]:
# Same wrapper and helpers as 02b (predict() fires at the 5% FPR operating point; see README ExpGrad setup)
class FixedFPRXGB(BaseEstimator, ClassifierMixin):
    def __init__(self, fpr=0.05, params=None):
        self.fpr, self.params = fpr, params
    def fit(self, X, y, sample_weight=None):
        self.classes_ = np.array([0, 1])
        self.model_ = train_model(X, y, sample_weight=sample_weight, params=self.params or BEST_PARAMS)
        self.cut_ = threshold_at_fpr(self.model_.predict_proba(X)[:, 1], y, self.fpr)
        return self
    def score_(self, X): return self.model_.predict_proba(X)[:, 1]
    def predict(self, X): return (self.score_(X) >= self.cut_).astype(int)

def mixture_score(eg, w, X):
    s = np.zeros(len(X))
    for wi, p in zip(w, eg.predictors_):
        if wi > 1e-8: s += wi * p.score_(X)
    return s / w[w > 1e-8].sum()

In [ ]:
# FIT: ONE fit at the Base-winning bound 0.05 (NOT re-swept). Expect ~1 h. Components are saved straight after the fit.
import time; t0 = time.time()
eg = ExponentiatedGradient(estimator=FixedFPRXGB(fpr=0.05),
                           constraints=FalsePositiveRateParity(difference_bound=0.05), max_iter=40)
eg.fit(Xc, yc, sensitive_features=gc)
w = np.asarray(eg.weights_, dtype=float); kept = [i for i, wi in enumerate(w) if wi > 1e-8]
print(f"fitted in {(time.time()-t0)/60:.1f} min; {len(eg.predictors_)} components, {len(kept)} non-zero")
for i in kept: eg.predictors_[i].model_.save_model(f"{OUT}/comp_{i}.json")
pd.DataFrame({"component": kept, "weight": w[kept]}).to_csv(f"{OUT}/weights.csv", index=False)

In [ ]:
# EVALUATE on Variant II months 6-7. Rows: BEFORE (frozen Base ExpGrad, Base cutoff), CONTROL (frozen, cutoff re-set on
# Variant II val, no retraining), AFTER (retrained on combined data, cutoff on Variant II val). Paste this output to Claude.
def metrics(y, g, dec):
    y, g, dec = map(np.asarray, (y, g, dec))
    f = [dec[(y == 0) & (g == k)].mean() for k in (0, 1)]
    return dict(recall=dec[y == 1].mean(), fpr=dec[y == 0].mean(), fpr_ratio=min(f) / max(f),
                recall_u50=dec[(y == 1) & (g == 0)].mean(), recall_50p=dec[(y == 1) & (g == 1)].mean())
Xv, yv, Xt, yt, gt = features(v2["val"]), v2["val"][LABEL], features(v2["test"]), v2["test"][LABEL], v2["test"]["age_group"]

ew = pd.read_csv(f"{ROOT}/models/expgrad_weights.csv")   # frozen Base ExpGrad 0.05 (verified earlier)
def lm(path):
    m = xgb.XGBClassifier(enable_categorical=True); m.load_model(path); return m
frozen = [(r.weight, lm(f"{ROOT}/models/expgrad_components/comp_{int(r.component)}.json")) for r in ew.itertuples()]
fscore = lambda X: sum(wt * m.predict_proba(X)[:, 1] for wt, m in frozen) / ew.weight.sum()
pb = pd.read_parquet(f"{PREDS}/preds_expgrad.parquet"); pbv = pb[pb.month == 5]
base_cut = threshold_at_fpr(pbv.score, pbv.y_true)

rows = {
 "BEFORE frozen, Base cutoff":      (fscore(Xt) >= base_cut).astype(int),
 "CONTROL frozen, recal on V2 val": (fscore(Xt) >= threshold_at_fpr(fscore(Xv), yv)).astype(int),
 "AFTER retrained, cutoff V2 val":  (mixture_score(eg, w, Xt) >= threshold_at_fpr(mixture_score(eg, w, Xv), yv)).astype(int),
}
print(pd.DataFrame({k: metrics(yt, gt, d) for k, d in rows.items()}).T.round(4))